# Compositional router: training data path (multi-benchmark)

This notebook mirrors **`notebooks/fine_grained_mcts_data_inspection.ipynb`**, but for the **first-order compositional router** (`routers/compositional_router.py`, `training/train_compositional_router.py`).

## Overlap with **fine** layer routing

| Aspect | Fine router (`build_fine_routing_dataset.py`) | Compositional router (`build_compositional_catalogues.py`) |
|--------|-----------------------------------------------|------------------------------------------------------------|
| **Action space** | Layer **permutations** / local edits around an anchor sequence | **DSL programs** (skip / repeat / swap / …) built from primitives |
| **Supervision signal** | Per-question **MCTS over layer sequences** → `explored` seqs + `router_target` softmax | **Observed (program_idx, delta)** pairs → softmax CE on **subset** of legal programs |
| **Geometry** | `FineRoutingConfig`: pivot, `editable_start`, swap radius | Manifest `geometry`: K, `swap_radius`, `editable_start`, assign flags |
| **Shared upstream** | Anchor + pivot residuals **concept** | Canonical `{bench}_pivot_residuals.pt` + programs; **separate** catalogue tree (`incidence/`, `observed/`, `primitives.jsonl`) |
| **Training loader** | `load_bench_data_mcts` etc. in `sweep_fine_routing.py` | `load_artifacts` + `CompositionalDataset` |

**Takeaway:** both can use similar **canonical** question-level inputs, but **schemas and targets differ**: fine routing predicts **which layer route**; compositional predicts **primitive/program** scores via sparse **A** (and optional pairs **B**).

## Setup

- Jupyter from **repo root** (first cell adjusts cwd).
- **`CATALOGUE_DIR`**: directory with `manifest.json` from `python -m data_prep.build_compositional_catalogues`.
- Env **`COMPOSITIONAL_BENCHMARKS`**: optional comma-separated subset; else all benchmarks in the manifest are loaded.
- Pivot `.pt` paths in the manifest are usually **repo-root-relative**.


## 1. Repository root


In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT:", REPO_ROOT)


REPO_ROOT: /home/janerik/flexible-test-time-program-selection


## 2. Catalogue directory and benchmarks


In [20]:
# Manifest uses "commonsenseqa" as the benchmark key (not "csqa").
os.environ["COMPOSITIONAL_BENCHMARKS"] = "commonsenseqa"

CATALOGUE_DIR = Path(
    os.environ.get(
        "COMPOSITIONAL_CATALOGUE_DIR",
        str(REPO_ROOT / "compositional_runs/joint5_bundle_noassign_compositional"),
    )
)
_bench_env = os.environ.get("COMPOSITIONAL_BENCHMARKS", "").strip()
BENCHMARKS = None
if _bench_env:
    BENCHMARKS = [b.strip() for b in _bench_env.split(",") if b.strip()]

print("CATALOGUE_DIR:", CATALOGUE_DIR)
print("BENCHMARKS filter:", BENCHMARKS or "<all in manifest>")


CATALOGUE_DIR: /home/janerik/flexible-test-time-program-selection/compositional_runs/joint5_bundle_noassign_compositional
BENCHMARKS filter: ['commonsenseqa']


In [13]:
print(_bench_env)

csqa


## 3. `load_artifacts`

Same as `training/train_compositional_router.py`.


In [21]:
from routers.compositional_router import CompositionalArtifacts, load_artifacts

artifacts: CompositionalArtifacts = load_artifacts(CATALOGUE_DIR, benchmarks=BENCHMARKS)

print("Loaded benchmarks:", artifacts.benchmarks)
print("|primitives| (M):", artifacts.num_primitives)
print("manifest keys:", sorted(artifacts.manifest.keys()))


Loaded benchmarks: ['commonsenseqa']
|primitives| (M): 28
manifest keys: ['M', 'benchmarks', 'data_dir', 'filter', 'geometry', 'output_dir', 'overall_stats', 'primitives_path']


## 4. Manifest geometry + filter


In [22]:
print(json.dumps(artifacts.manifest.get("geometry"), indent=2))
print(json.dumps(artifacts.manifest.get("filter"), indent=2))


{
  "K": 2,
  "swap_radius": 3,
  "editable_start": 17,
  "include_assign": false,
  "dedupe_assign_with_struct": false
}
{
  "min_count": 1,
  "min_questions": 1,
  "min_benchmarks": 1,
  "keep_kinds": [
    "skip",
    "repeat",
    "swap"
  ],
  "support_path": "compositional_runs/joint5_bundle_noassign/primitive_support.jsonl"
}


## 5. Per-benchmark catalogue stats

`LegalCatalogue`: sparse **A** [programs × primitives]; dense **lengths** is **ℓ** in `S = A u + B v − λ ℓ`.


In [23]:
rows = []
for b in artifacts.benchmarks:
    cat = artifacts.catalogues[b]
    info = artifacts.manifest["benchmarks"][b]
    rows.append(
        {
            "benchmark": b,
            "n_programs": cat.n_programs,
            "n_primitives": cat.n_primitives,
            "has_pairs": cat.has_pairs,
            "n_pairs": cat.n_pairs,
            "manifest_n_legal": info.get("n_legal_programs"),
            "observed_path": info.get("observed_path"),
        }
    )

for r in rows:
    print(r)


{'benchmark': 'commonsenseqa', 'n_programs': 206, 'n_primitives': 28, 'has_pairs': True, 'n_pairs': 180, 'manifest_n_legal': 206, 'observed_path': 'observed/commonsenseqa.jsonl'}


## 6. `primitives.jsonl` sample


In [24]:
prim_path = CATALOGUE_DIR / artifacts.manifest["primitives_path"]
with open(prim_path) as f:
    prim_lines = [f.readline() for _ in range(5)]
prim_sample = [json.loads(x) for x in prim_lines if x.strip()]
print("path:", prim_path)
for p in prim_sample:
    print({k: p[k] for k in ("idx", "kind", "args", "key") if k in p})


path: /home/janerik/flexible-test-time-program-selection/compositional_runs/joint5_bundle_noassign_compositional/primitives.jsonl
{'idx': 0, 'kind': 'skip', 'args': [17], 'key': 'skip(17)'}
{'idx': 1, 'kind': 'repeat', 'args': [17], 'key': 'repeat(17)'}
{'idx': 2, 'kind': 'swap', 'args': [17, 18], 'key': 'swap(17,18)'}
{'idx': 3, 'kind': 'swap', 'args': [17, 19], 'key': 'swap(17,19)'}
{'idx': 4, 'kind': 'swap', 'args': [17, 20], 'key': 'swap(17,20)'}


## 7. Legal + observed first row

**Observed** drives supervision: `obs_indices` (program row ids), `obs_deltas`.


In [25]:
for b in artifacts.benchmarks[:5]:
    info = artifacts.manifest["benchmarks"][b]
    lp = CATALOGUE_DIR / info["legal_programs_path"]
    obs = CATALOGUE_DIR / info["observed_path"]
    with open(lp) as f:
        legal0 = json.loads(f.readline())
    with open(obs) as f:
        obs0 = json.loads(f.readline())
    print("===", b, "===")
    print("legal0 keys:", sorted(legal0.keys()))
    print("obs0 keys:", sorted(obs0.keys()))
    print("obs0 n_obs:", obs0.get("n_obs", len(obs0.get("obs_indices", []))))
    print("obs slice (idx, delta):", list(zip(obs0["obs_indices"][:6], obs0["obs_deltas"][:6])))
    print()


=== commonsenseqa ===
legal0 keys: ['idx', 'key', 'length', 'primitive_indices']
obs0 keys: ['dropped_program_entries', 'n_obs', 'obs_deltas', 'obs_indices', 'question_hash', 'question_id', 'residual_idx']
obs0 n_obs: 19
obs slice (idx, delta): [(0, 0.0), (1, 0.380859375), (2, 1.26171875), (3, 1.177734375), (7, 0.640625), (9, -1.25)]



## 8. Pivot residuals (name overlap with fine router only)

Resolve path vs repo root; load tensor shape.


In [26]:
import torch

b0 = artifacts.benchmarks[0]
info0 = artifacts.manifest["benchmarks"][b0]
pivot_rel = info0["pivot_residuals_path"]
pivot_path = Path(pivot_rel)
if not pivot_path.is_file():
    pivot_path = REPO_ROOT / pivot_rel
print("benchmark:", b0)
print("pivot_residuals_path:", pivot_path)
print("exists:", pivot_path.is_file())
if pivot_path.is_file():
    t = torch.load(pivot_path, map_location="cpu", weights_only=True).float()
    print("tensor shape:", tuple(t.shape), t.dtype)


benchmark: commonsenseqa
pivot_residuals_path: compositional_runs/joint5_bundle_noassign/commonsenseqa_pivot_residuals.pt
exists: True
tensor shape: (9741, 896) torch.float32


## 9. `CompositionalDataset` — joint multi-benchmark

Concatenates questions from all `benchmarks`; each item has `benchmark_id` for routing **A** in the trainer. Matches `--scope joint`.


In [27]:
from routers.compositional_router import CompositionalDataset

bench_to_id = {b: i for i, b in enumerate(sorted(artifacts.benchmarks))}
print("bench_to_id:", bench_to_id)

dataset = CompositionalDataset(
    artifacts,
    benchmarks=artifacts.benchmarks,
    bench_to_id=bench_to_id,
)
print("len(dataset):", len(dataset))


bench_to_id: {'commonsenseqa': 0}
len(dataset): 9287


In [30]:
sample = dataset[0]
print("sample keys:", sorted(sample.keys()))
bid = sample["benchmark_id"]
print("benchmark:", sample["benchmark"], "benchmark_id:", bid.item() if hasattr(bid, "item") else bid)
print("question_id:", sample["question_id"])
print("encoder_input shape:", tuple(sample["encoder_input"].shape))
print("obs_indices shape:", tuple(sample["obs_indices"].shape))
print("obs_deltas shape:", tuple(sample["obs_deltas"].shape))


sample keys: ['benchmark', 'benchmark_id', 'encoder_input', 'obs_deltas', 'obs_indices', 'question_id']
benchmark: commonsenseqa benchmark_id: 0
question_id: 0
encoder_input shape: (896,)
obs_indices shape: (19,)
obs_deltas shape: (19,)


## 10. Optional: filtered observed (`build_mcts_filtered_observed`)

Set **`COMPOSITIONAL_FILTERED_OBSERVED_DIR`** to the output dir that contains `observed/{bench}.jsonl` from that script. **A** / **ℓ** stay full-catalogue; only supervision rows change.


In [29]:
FILTERED_OBS_BASE = os.environ.get("COMPOSITIONAL_FILTERED_OBSERVED_DIR", "")
if FILTERED_OBS_BASE:
    base = Path(FILTERED_OBS_BASE)
    overrides = {}
    for b in artifacts.benchmarks:
        p = base / "observed" / f"{b}.jsonl"
        if p.is_file():
            overrides[b] = p
    print("overrides:", overrides)
    ds_f = CompositionalDataset(
        artifacts,
        benchmarks=list(overrides.keys()),
        bench_to_id=bench_to_id,
        observed_path_overrides=overrides,
    )
    print("filtered len:", len(ds_f))
else:
    print("Set COMPOSITIONAL_FILTERED_OBSERVED_DIR to try filtered observed loads.")


Set COMPOSITIONAL_FILTERED_OBSERVED_DIR to try filtered observed loads.
